1. Convert PDF files into document data structure

In [1]:
from langchain_community.document_loaders import PyMuPDFLoader
from pathlib import Path

def PDFs_to_Documents(path_directory):
    """A function to convert all PDF files in the given directory to Documents."""
    directory = Path(path_directory)
    PDF_files = list(directory.glob("**/*.pdf"))
    print(f"Found {len(PDF_files)} PDF files in the directory.")
    print()
    all_documents = []
    for PDF_file in PDF_files:
        print(f"Processing: {PDF_file.name}")
        try:
            loader = PyMuPDFLoader(str(PDF_file))
            documents = loader.load()
            for document in documents:
                document.metadata["source_file"] = PDF_file.name
                document.metadata["file_type"] = "pdf"
                document.metadata["author"] = "Nupur Maheshwari"
                document.metadata["date_created"] = "2026-10-20"
            all_documents.extend(documents)
            print(f"Loaded {len(documents)} documents from {PDF_file.name}")
            print()
        except Exception as e:
            print(f"Error occured while processing {PDF_file.name} : {e}")
            print()
    
    print(f"Total documents loaded: {len(all_documents)}")
    print()
    return all_documents

all_PDFs_to_Documents = PDFs_to_Documents("Data")
all_PDFs_to_Documents

C:\Users\BIT\AppData\Local\Temp\ipykernel_14168\460513127.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


Found 2 PDF files in the directory.

Processing: ChocoKari_Info.pdf
Loaded 5 documents from ChocoKari_Info.pdf

Processing: Nupur_Resume.pdf
Loaded 1 documents from Nupur_Resume.pdf

Total documents loaded: 6



[Document(metadata={'producer': 'Skia/PDF m156 Google Docs Renderer', 'creator': '', 'creationdate': '', 'source': 'Data\\PdfFiles\\ChocoKari_Info.pdf', 'file_path': 'Data\\PdfFiles\\ChocoKari_Info.pdf', 'total_pages': 5, 'format': 'PDF 1.4', 'title': 'ChocoKari', 'author': 'Nupur Maheshwari', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0, 'source_file': 'ChocoKari_Info.pdf', 'file_type': 'pdf', 'date_created': '2026-10-20'}, page_content="CHOCOKARI — COMPLETE KNOWLEDGE BASE \n \n1. BRAND & BUSINESS OVERVIEW \nBrand Name: ChocoKari \nStory: Born in a home kitchen, perfected in every batch. Each piece is hand-tempered, blended with the finest \ninclusions, and finished to a silky shine. Small-batch chocolate made for life's sweetest moments. \nTagline: Every Bite, A Story to Savor \nBusiness Type: Handcrafted artisan chocolate, made in small batches in India. \nMission: Premium handcrafted chocolates using the world's finest in

2. Split all documents into chunks

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def Documents_to_Chunks(documents, chunk_size=600, chunk_overlap=150):
    """A function to split all the documents into chunks."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    all_chunks = splitter.split_documents(documents)
    print(f"{len(documents)} documents split into {len(all_chunks)} chunks.")
    print()
    return all_chunks

all_Documents_to_Chunks = Documents_to_Chunks(all_PDFs_to_Documents)
all_Documents_to_Chunks

6 documents split into 35 chunks.



[Document(metadata={'producer': 'Skia/PDF m156 Google Docs Renderer', 'creator': '', 'creationdate': '', 'source': 'Data\\PdfFiles\\ChocoKari_Info.pdf', 'file_path': 'Data\\PdfFiles\\ChocoKari_Info.pdf', 'total_pages': 5, 'format': 'PDF 1.4', 'title': 'ChocoKari', 'author': 'Nupur Maheshwari', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0, 'source_file': 'ChocoKari_Info.pdf', 'file_type': 'pdf', 'date_created': '2026-10-20'}, page_content="CHOCOKARI — COMPLETE KNOWLEDGE BASE \n \n1. BRAND & BUSINESS OVERVIEW \nBrand Name: ChocoKari \nStory: Born in a home kitchen, perfected in every batch. Each piece is hand-tempered, blended with the finest \ninclusions, and finished to a silky shine. Small-batch chocolate made for life's sweetest moments. \nTagline: Every Bite, A Story to Savor \nBusiness Type: Handcrafted artisan chocolate, made in small batches in India. \nMission: Premium handcrafted chocolates using the world's finest in

3. Make class for converting chunks to embeddings

In [3]:
import numpy as np
from sentence_transformers import SentenceTransformer

class EmbeddingsGenerator:
    def __init__(self, model_name : str):
        self.model_name = model_name
        self.model = SentenceTransformer(model_name,device="cpu")
        print(f"{self.model_name} model loaded successfully with dimension: {self.model.get_sentence_embedding_dimension()}")
        print()

    def generate_embedding(self, text : list[str]) -> np.ndarray:
        if self.model is None:
            raise ValueError(f"{self.model_name} model not loaded")
        print(f"Generating embeddings for {len(text)} texts...")
        print()
        embeddings = self.model.encode(
            text,
            batch_size=1,
            show_progress_bar=True,
            convert_to_numpy=True
        )
        print(f"Generated embeddings with shape: {embeddings.shape}")
        print()
        return embeddings

embeddings_generator = EmbeddingsGenerator("all-MiniLM-L6-v2")
embeddings_generator

all-MiniLM-L6-v2 model loaded successfully with dimension: 384



4. Make class for storing generated embeddings in vector database

In [4]:
import os
import chromadb
import uuid
from typing import List, Dict, Any

class VectorStore:
    def __init__(self, collection_name : str, persist_directory : str):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description" : "PDF Document embeddings for RAG"}
            )
            print(f"Vector store initialized.")
            print()
            print(f"Existing collections in the vector store {self.collection_name}: {self.collection.count()}")
            print()
        except Exception as e:
            print(f"Error occured while initializing vector store {self.collection_name} : {e}")
            print()

    def add_documents_embeddings(self, documents : list[Any], embeddings : np.ndarray):
        if len(documents) != len(embeddings):
            raise ValueError("The number of documents must match the number of embeddings.")
        print(f"Adding {len(documents)} records to the vector store...")
        print()
        id_list = []
        page_content_list = []
        metadata_list = []
        embedding_list = []
        for i, (document, embedding) in enumerate(zip(documents,embeddings)):
            id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            id_list.append(id)
            page_content_list.append(document.page_content)
            metadata = dict(document.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(document.page_content)
            metadata_list.append(metadata)
            embedding_list.append(embedding.tolist())
        try:
            self.collection.add(
                ids=id_list,
                documents=page_content_list,
                metadatas=metadata_list,
                embeddings=embedding_list
            )
            print(f"Successfully added {len(documents)} records to the vector store.")
            print()
            print(f"Existing collections in the vector store {self.collection_name}: {self.collection.count()}")
            print()
        except Exception as e:
            print(f"Error occured while adding records to the vector store")
            print()
            
vector_store = VectorStore("pdf_RAG_embeddings","VectorStore")
vector_store

Vector store initialized.

Existing collections in the vector store pdf_RAG_embeddings: 35



In [5]:
texts = [chunk.page_content for chunk in all_Documents_to_Chunks]

embeddings = embeddings_generator.generate_embedding(texts)

vector_store.add_documents_embeddings(all_Documents_to_Chunks,embeddings)

Generating embeddings for 35 texts...



Batches:   0%|          | 0/35 [00:00<?, ?it/s]

Generated embeddings with shape: (35, 384)

Adding 35 records to the vector store...

Successfully added 35 records to the vector store.

Existing collections in the vector store pdf_RAG_embeddings: 35



5. Make class for retrieving results similar to the query

In [5]:
class RAGretriever:
    def __init__(self, embeddings_generator : EmbeddingsGenerator, vector_store : VectorStore):
        self.embeddings_generator = embeddings_generator
        self.vector_store = vector_store

    def retrieve_results(self, query : str, top_k : int, score_threshold : float) -> List[Dict[str,Any]]:
        print(f"Retrieving top {top_k} results with score greater than {score_threshold} for query: {query}")
        print()
        query_embedding = self.embeddings_generator.generate_embedding([query])[0]
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            retrieved_results = []
            if results["documents"] and results["documents"][0]:
                ids = results["ids"][0]
                documents = results["documents"][0]
                metadatas = results["metadatas"][0]
                distances = results["distances"][0]
                for i, (id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    similarity_score = 1 - distance
                    if similarity_score >= score_threshold:
                        retrieved_results.append({
                            "id" : id,
                            "page_content" : document,
                            "metadata" : metadata,
                            "similarity_score" : similarity_score,
                            "distance" : distance,
                            "rank" : i + 1
                        })
                print(f"Retrived {len(retrieved_results)} similar results for the query - {query}")
                print()
            else:
                print(f"No results found similar to the query - {query}")
                print()
            return retrieved_results
        except Exception as e:
            print(f"Error occured while retrieving results similar to the query - {query} : {e}")
            print()

rag_retriever = RAGretriever(embeddings_generator, vector_store)
rag_retriever

In [6]:
results = rag_retriever.retrieve_results("Give summary of Nupur Maheshwari in BTech AI ML",3,0.0)
results

Retrieving top 3 results with score greater than 0.0 for query: Give summary of Nupur Maheshwari in BTech AI ML

Generating embeddings for 1 texts...



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)

Retrived 1 similar results for the query - Give summary of Nupur Maheshwari in BTech AI ML



[{'id': 'doc_16b07f91_22',
  'page_content': 'NUPUR MAHESHWARI \nMumbai, Maharashtra  |  Email: nupur.maheshwari.2010@gmail.com  |  Phone: 7977316546  |  LinkedIn  |  GitHub | Portfolio \nSUMMARY \n \nB.Tech AI & ML student (CGPA: 9.37) with practical software development experience building and deploying full-stack applications using the MERN stack. \nStrong in secure backend architecture, REST APIs, authentication, real-time systems, and AI-integrated features, with experience owning applications',
  'metadata': {'producer': 'Skia/PDF m156 Google Docs Renderer',
   'trapped': '',
   'keywords': '',
   'creationdate': '',
   'source': 'Data\\PdfFiles\\Nupur_Resume.pdf',
   'author': 'Nupur Maheshwari',
   'file_type': 'pdf',
   'moddate': '',
   'creator': '',
   'subject': '',
   'doc_index': 22,
   'page': 0,
   'total_pages': 1,
   'date_created': '2026-10-20',
   'title': 'Nupur_Resume',
   'modDate': '',
   'file_path': 'Data\\PdfFiles\\Nupur_Resume.pdf',
   'content_length': 452

6. Make class for generating response for the query using similar context

In [7]:
from dotenv import load_dotenv

load_dotenv()

True

In [8]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage

class LLM:
    def __init__(self, model_name : str, api_key : str):
        self.model_name = model_name
        self.api_key = api_key
        self.llm = ChatGroq(model=self.model_name,temperature=0.1,max_tokens=1024)
        print(f"Initialized Groq LLM with model: {self.model_name}")
        print()

    def generate_response(self, context : str, query : str) -> str:
        prompt_template = PromptTemplate(
            input_variables=["context","question"],
            template="""
            You are a helpful AI assisstent. Use the following context to answer the question accurately and concisely.
            Context:
            {context}
            Question:
            {question}
            Answer:
            Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so.
            """
        )
        prompt_formatted = prompt_template.format(context=context,question=query)
        try:
            messages = [HumanMessage(content=prompt_formatted)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            print(f"Error occured while generating response by Groq LLM model {self.model_name} : {e}")
            print()

llm = LLM("openai/gpt-oss-20b",os.getenv("GROQ_API_KEY"))
llm

Initialized Groq LLM with model: openai/gpt-oss-20b



In [9]:
context = "\n\n".join([result["page_content"] for result in results]) if results else ""
answer = llm.generate_response(context,"Give summary of Nupur Maheshwari of AI ML")
answer

'**Nupur Maheshwari – AI & ML Profile**\n\n- **Education**: B.Tech in Artificial Intelligence & Machine Learning (CGPA\u202f9.37).  \n- **Technical Expertise**:  \n  - Full‑stack development with the MERN stack (MongoDB, Express, React, Node.js).  \n  - Secure backend architecture, RESTful APIs, authentication mechanisms, and real‑time system design.  \n  - Integration of AI/ML features into production applications.  \n- **Experience**: Hands‑on software development, end‑to‑end deployment, and ownership of complete application lifecycles.  \n\nNupur combines strong academic credentials with practical, industry‑ready skills in building secure, AI‑enhanced web applications.'

TO CLEAR VECTOR DB

vector_store.client.delete_collection(name=vector_store.collection_name)